# Agentic Scientific Discovery — Demo

This notebook sketches the **architecture** and runs a **short autonomous investigation** on the synthetic gene-expression cohort (same as `examples/gene_expression_investigation/run.py`).

**Prerequisites:** from the project root, `pip install -r requirements.txt`, `export PYTHONPATH=.`, and optionally `export MPLBACKEND=Agg` for headless plotting.

## System architecture (conceptual)

```
KnowledgeStore + ExperimentLog + Retriever
          ▲                 │
          │                 │ context
          │                 ▼
   ResultAnalyzer ◄── aggregates + numeric digest
          ▲
          │ traces
   ExperimentExecutor ──► tools/ (stats, viz, PubMed, STRING, …)
          ▲
          │ ExperimentPlan (JSON / dataclass)
   ExperimentPlanner
          ▲
   HypothesisGenerator ──► ranked by information gain / feasibility
          ▲
   ResearchOrchestrator (budget, strategy pulse every N cycles)
```

**Design goal:** every claim is backed by structured records (hypothesis → plan → trace → interpretation → finding).

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").is_dir():
    PROJECT = ROOT
else:
    PROJECT = ROOT.parent
sys.path.insert(0, str(PROJECT))
os.environ.setdefault("MPLBACKEND", "Agg")

print("PROJECT:", PROJECT)

In [ ]:
from pathlib import Path

import sys

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.agent.orchestrator import OrchestratorConfig, ResearchBudget, ResearchOrchestrator
from src.config import create_llm_backend, get_config
from src.memory.experiment_log import ExperimentLog
from src.memory.knowledge_store import KnowledgeStore
from src.tools.data_analysis import ToolContext, profile_dataset

sys.path.insert(0, str(PROJECT / "examples" / "gene_expression_investigation"))
from synthetic_cohort import build_synthetic_cohort

cfg = get_config()
session_dir = PROJECT / "examples" / "gene_expression_investigation" / "notebook_session"
session_dir.mkdir(parents=True, exist_ok=True)
knowledge_dir = session_dir / "knowledge"
log_path = session_dir / "experiments.json"
output_dir = session_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

expr, groups, pathway_sets = build_synthetic_cohort(seed=7)


def ctx_factory():
    return ToolContext(
        expression=expr.copy(),
        groups=groups.copy(),
        pathway_sets=pathway_sets,
        output_dir=output_dir,
    )


def dataset_summary():
    ctx = ctx_factory()
    return {**profile_dataset(ctx, {}), "scenario": "synthetic_planted_pathway"}


llm = create_llm_backend(cfg)
knowledge = KnowledgeStore(knowledge_dir)
experiments = ExperimentLog(log_path)

orch = ResearchOrchestrator(
    llm=llm,
    knowledge=knowledge,
    experiments=experiments,
    tool_ctx_factory=ctx_factory,
    orch_cfg=OrchestratorConfig(
        research_question=(
            "What biological processes distinguish the treatment group from control?"
        ),
        available_data_keys=["expression_matrix", "sample_groups", "pathway_gene_sets"],
        budget=ResearchBudget(max_cycles=3, max_wall_time_s=None),
        strategy_every_n_cycles=2,
    ),
    dataset_summary_provider=dataset_summary,
    config=cfg,
)

cycles = orch.run()
orch.save_session(session_dir)
print("Cycles:", len(cycles))

In [ ]:
from pprint import pprint

for c in cycles:
    print("=== cycle", c["cycle"], "===")
    for d in c["decisions"]:
        step = d.get("step")
        if step in {"review_knowledge", "execute", "strategy"}:
            print(step, "keys:", list(d.keys()))
        elif step == "rank_hypotheses":
            top = d["ranking"][0] if d["ranking"] else None
            print(step, "top:", top)
        elif step == "analyze":
            print(step, d["analysis"]["verdict"], "post", round(d["analysis"]["posterior"], 3))
        elif step == "plan":
            print(step, "n_steps", len(d["plan"]["steps"]))

In [ ]:
import pandas as pd

pe_path = output_dir / "pathway_enrichment.csv"
if pe_path.exists():
    display(pd.read_csv(pe_path).head(10))
else:
    print("Run enrichment step first (pathway_enrichment.csv).")

## Knowledge graph (lightweight)

The knowledge store tracks **hypotheses → experiments → findings**. For a quick graph view, link entities by parsing `hypothesis_id` / `experiment_id` fields in `knowledge/findings.json` (at scale you’d promote this to a real graph DB or RDF). The **provenance chain** is the portfolio story: no floating claims.

## Final research summary (template)

In a full deployment you’d render the latest findings, open questions, and confidence posteriors into a LaTeX / HTML report. Here, inspect:

- `notebook_session/knowledge/findings.json` — evidence‑strength + links back to experiments  
- `notebook_session/experiments.json` — full tool traces for reproducibility  

### What would I change at scale?

- **GraphRAG** over genes, pathways, and papers rather than flat JSON.  
- **Typed tool schemas** (e.g. Pydantic / JSON Schema) returned by the planner, not free‑formed dicts.  
- **Hierarchical policies**: lab safety / statistician review gates between cycles.  
- **Closed‑loop lab**: swap `ExperimentExecutor` for an instrument driver with the same provenance headers.